# Phase 4: Causal Patching with nnsight

This notebook will take your cleanly labeled pairs (clean vs counterfactual) and use `nnsight` to perform surgical intervention on Layer 10 of Llama-3.2-3B. We will prove that injecting the hallucinating hidden state causally forces the model to derail.

In [ ]:
!pip install -q nnsight transformers accelerate huggingface_hub
import huggingface_hub
huggingface_hub.login(token="YOUR_HF_TOKEN") # REPLACE WITH YOUR TOKEN

## 1. Load Model and Data

In [ ]:
import json
import torch
from tqdm.auto import tqdm
from nnsight import LanguageModel
from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download

print("Loading Llama-3.2-3B via nnsight...")
model = LanguageModel("meta-llama/Llama-3.2-3B-Instruct", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print("Downloading dataset from Hugging Face...")
file_path = hf_hub_download(repo_id="AnishRacherla/LinguaFranca-Phase3", filename="data/2wikimultihopqa/augmented.jsonl", repo_type="dataset")

with open(file_path) as f:
    data = [json.loads(line) for line in f]
    
counterfactuals = [d for d in data if d.get("is_counterfactual", False)]
cleans = {d["id"]: d for d in data if not d.get("is_counterfactual", False)}

print(f"Loaded {len(cleans)} clean examples and {len(counterfactuals)} counterfactuals.")

## 2. Prepare Perfect Minimal Pairs

In [ ]:
# Find pairs where clean succeeds and CF fails
pairs = []
for cf in counterfactuals:
    clean = cleans.get(cf["clean_pair_id"])
    if not clean: continue
    
    clean_h1 = next((h for h in clean.get("hops", []) if h["hop_idx"] == 1), None)
    cf_h1 = next((h for h in cf.get("hops", []) if h["hop_idx"] == 1), None)
    
    if clean_h1 and cf_h1 and clean_h1.get("label") == 0 and cf_h1.get("label") == 1:
        pairs.append((clean, cf))

print(f"Found {len(pairs)} perfect minimal pairs for causal patching.")

## 3. Perform Causal Brain Surgery!

In [ ]:
target_layer = 10 # From Phase 3 results!
successful_interventions = 0
total_run = min(len(pairs), 78) # Limit if you want

if total_run == 0:
    print("No perfect pairs found!")
else:
    for i in tqdm(range(total_run), desc="Performing Brain Surgery"):
        clean, cf = pairs[i]
        clean_prompt = clean["prompt"]
        cf_prompt = cf["prompt"]
        
        # We will run both through the model, and intervene on Layer 10!
        with model.trace() as tracer:
            # 1. Trace the counterfactual to harvest its hallucinating brain waves
            with tracer.invoke(cf_prompt):
                # Save the hidden state at layer 10 for the LAST token of the prompt
                # (This is the moment right before it starts reasoning)
                cf_hidden_state = model.model.layers[target_layer].output[0][:, -1, :].save()
                
            # 2. Trace the clean prompt and inject the hallucinating brain waves!
            with tracer.invoke(clean_prompt):
                # Overwrite the clean model's Layer 10 hidden state at the last token
                # with the harvested state from the counterfactual!
                model.model.layers[target_layer].output[0][:, -1, :] = cf_hidden_state
                
                # Generate the corrupted output
                corrupted_output = model.generator(max_new_tokens=100)

        generated_text = tokenizer.decode(corrupted_output.save().value[0][-100:], skip_special_tokens=True)
        
        # Check if we successfully corrupted the model
        if cf["substitute"].lower() in generated_text.lower():
            successful_interventions += 1
            
        if i == 0:
            print("=== Example Intervention ===")
            print("\n--- Clean Question ---")
            print(clean["question"])
            print("\n--- Counterfactual Question ---")
            print(cf["question"])
            print("\n--- Corrupted Generation (Should hallucinate!) ---")
            print(generated_text)
            print("============================\n")
            
    print(f"\nCausal Patching Complete!")
    print(f"Successfully forced the healthy model to hallucinate in {successful_interventions} out of {total_run} pairs!")
    print(f"Causal Corruption Rate: {(successful_interventions/total_run)*100:.1f}%")